In [1]:
import optuna
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [2]:
df = pd.read_csv('reddit_preprocessing.csv').dropna()
df.shape

(36662, 2)

In [ ]:
import matplotlib.pyplot as plt
import optuna
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df["category"] = df["category"].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=["category"])

ngram_range = (1, 3)  # Trigram setting
max_features = 10000  # Set max_features to 10000 for TF-IDF

# Step 4: Train-test split before vectorization and resampling
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_comment"],
    df["category"],
    test_size=0.2,
    random_state=42,
    stratify=df["category"],
)

# Step 2: Vectorization using TF-IDF, fit on training data only
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train_vec = vectorizer.fit_transform(X_train)  # Fit on training data
X_test_vec = vectorizer.transform(X_test)  # Transform test data

smote = SMOTE(random_state=42)
X_train_vec, y_train = smote.fit_resample(X_train_vec, y_train)


# Local Evaluation Function replacing MLflow tracking
def evaluate_and_report(model_name, model, X_train, X_test, y_train, y_test):
    # Train model
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)

    # Print a structured report to the console
    print("=" * 60)
    print(f"FINAL OPTIMIZED MODEL EVALUATION: {model_name}")
    print("=" * 60)
    print(f"Accuracy: {accuracy:.4f}\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print("=" * 60)

    # Optional: Generate and display a local confusion matrix plot
    conf_matrix = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix: Best Optimized {model_name}")
    plt.savefig(f"best_{model_name.lower()}_confusion_matrix.png")
    plt.show()

    return model


# Step 6: Optuna objective function for XGBoost
def objective_xgboost(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int("max_depth", 3, 10)

    model = XGBClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        random_state=42,
    )
    # Fit and score on your validation split
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_test_vec)
    return accuracy_score(y_test, preds)


# Step 7: Run Optuna for XGBoost, evaluate the best model locally
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_xgboost, n_trials=30)

    # Print Optuna summary metrics
    print("\n##### OPTUNA TUNING COMPLETE #####")
    print(f"Best Trial Accuracy: {study.best_value:.4f}")
    print("Best Parameters:")
    for key, value in study.best_params.items():
        print(f"  - {key}: {value}")
    print("=" * 60 + "\n")

    # Initialize the best model using the ideal parameter set found by Optuna
    best_params = study.best_params
    best_model = XGBClassifier(
        n_estimators=best_params["n_estimators"],
        learning_rate=best_params["learning_rate"],
        max_depth=best_params["max_depth"],
        random_state=42,
    )

    # Route optimized model metadata directly to our local evaluation function
    evaluate_and_report("XGBoost", best_model, X_train_vec, X_test_vec, y_train, y_test)


# Run the experiment for XGBoost
run_optuna_experiment()


[I 2026-10-25 22:03:14,468] A new study created in memory with name: no-name-562108e0-9810-4fff-af70-48bf9be34654
[I 2026-10-25 22:07:02,286] Trial 0 finished with value: 0.6338469930451384 and parameters: {'n_estimators': 186, 'learning_rate': 0.00295800630188089, 'max_depth': 9}. Best is trial 0 with value: 0.6338469930451384.
[I 2026-10-25 22:13:36,396] Trial 1 finished with value: 0.6101186417564435 and parameters: {'n_estimators': 256, 'learning_rate': 0.0001032849012368234, 'max_depth': 10}. Best is trial 0 with value: 0.6338469930451384.
[I 2026-10-25 22:15:26,218] Trial 2 finished with value: 0.597708986772126 and parameters: {'n_estimators': 298, 'learning_rate': 0.0026853605245498063, 'max_depth': 5}. Best is trial 0 with value: 0.6338469930451384.
[I 2026-10-25 22:16:19,546] Trial 3 finished with value: 0.7254875221600982 and parameters: {'n_estimators': 235, 'learning_rate': 0.051729323457086356, 'max_depth': 4}. Best is trial 3 with value: 0.7254875221600982.
[I 2026-10-25


##### OPTUNA TUNING COMPLETE #####
Best Trial Accuracy: 0.8114
Best Parameters:
  - n_estimators: 266
  - learning_rate: 0.09090312892966049
  - max_depth: 9

